# Building a Neural netwok using single perceptron

In [1]:
import torch 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Library used for data preprocessing.

In [2]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate , KFold ,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report ,confusion_matrix

### This dataset is used for train and test our neural network.

In [3]:
df = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\Program\AI ML\heart.csv")
df.sample(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
740,54,F,NAP,160,201,0,Normal,163,N,0.0,Up,0
801,56,M,ASY,132,184,0,LVH,105,Y,2.1,Flat,1
868,51,M,NAP,110,175,0,Normal,123,N,0.6,Up,0
715,44,F,NAP,108,141,0,Normal,175,N,0.6,Flat,0
12,39,M,ATA,120,204,0,Normal,145,N,0.0,Up,0


In [4]:
categorical_column = ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
numric_columns = df.drop(columns=['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope','HeartDisease'] ,axis=1).columns.tolist()

### Numric data Pipeline

In [5]:
Numric_pipeline = Pipeline([
    ("SimpleImputer",SimpleImputer(strategy='median')),
    ("Scaler",StandardScaler())
])

### Categorical Data Pipeline.

In [6]:
categorical_pipeline =Pipeline([
    ("ohe",OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

### Full pipeline

In [7]:
complete_pipeline = ColumnTransformer([
    ("num",Numric_pipeline,numric_columns),
    ("cat",categorical_pipeline,categorical_column)
])

In [8]:
target = df['HeartDisease']
final_df = df.drop(['HeartDisease'],axis=1)

In [9]:
process_data = complete_pipeline.fit_transform(final_df)
process_data

array([[-1.4331398 ,  0.41090889,  0.82507026, ...,  0.        ,
         0.        ,  1.        ],
       [-0.47848359,  1.49175234, -0.17196105, ...,  0.        ,
         1.        ,  0.        ],
       [-1.75135854, -0.12951283,  0.7701878 , ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 0.37009972, -0.12951283, -0.62016778, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.37009972, -0.12951283,  0.34027522, ...,  0.        ,
         1.        ,  0.        ],
       [-1.64528563,  0.30282455, -0.21769643, ...,  0.        ,
         0.        ,  1.        ]])

In [10]:
process_data.shape , target.shape

((918, 20), (918,))

### Spliting data set for training and testing 
### Training = 70%
### Testing = 30%

In [11]:
x_train,x_test,y_train,y_test = train_test_split(process_data, target, random_state=42, test_size=0.3, stratify= target)

## Convert numpy data  into tensor data 

In [12]:
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test, dtype= torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy())
y_test_tesnor = torch.tensor(y_test.to_numpy())

# Let's build our model.

In [13]:
class perceptron:
    def __init__(self,lt = 1e-1):
        self.w = None
        self.b = None
        self.lt = lt

    #forward pass.
    def forward(self,X):
        z = torch.matmul(X,self.w) + self.b
        return torch.sigmoid(z)
    

    
    # Define Log loss Function.
    def log_loss(self, y_true, y_predicted):
        epsilon = 1e-10

        # Clamp values to avoid log(0)
        y_predicted = torch.clamp(y_predicted, epsilon, 1 - epsilon)
    
        loss = -torch.mean(y_true * torch.log(y_predicted) + (1 - y_true) * torch.log(1 - y_predicted))
        return loss



    # Define Fit Function.
    def fit(self, X ,Y, epochs):

        prev_loss = float('inf')
        accuracy = 0

        self.w = torch.rand(X.shape[1], dtype= torch.float32 ,requires_grad= True)
        self.b = torch.zeros(1, dtype = torch.float32, requires_grad=True)

        for i in range(epochs):
            # forward pass.
            y_pred = self.forward(X)

            # calculating loss.
            curr_loss = self.log_loss(Y,y_pred)

            # Claculating Accuracy.
            y_pred = torch.round(y_pred)
            accuracy = ((y_pred == Y).sum().item() * 100) / len(Y)

            curr_loss.backward()

            with torch.no_grad():
                self.w -= self.lt * self.w.grad
                self.b -= self.lt * self.b.grad

            self.w.grad.zero_()
            self.b.grad.zero_()

            if (prev_loss - curr_loss) < 1e-5:
                break

            prev_loss = curr_loss

            if i % 10 == 0:
                print(f"epoch : {i} || loss : {curr_loss:.4f} || accuracy : {accuracy:.4f}")
        return prev_loss.item(),accuracy


    
    #Define predict function.
    def predict(self,X):
        z = torch.matmul(X, self.w) + self.b
        return torch.sigmoid(z)
        # return self.forward(X)

        
    #Define evaluate function.
    def evaluate(self, X, Y):
        y_pred = self.predict(X)
        loss = self.log_loss(Y, y_pred)
        
        y_pred = torch.round(y_pred)
        accuracy = ((y_pred == Y).sum().item() * 100) / len(Y)
        return loss.item() ,accuracy


In [14]:
model = perceptron()


### Model training 

In [15]:
model.fit(x_train_tensor,y_train_tensor,1000)

epoch : 0 || loss : 1.2052 || accuracy : 55.2960
epoch : 10 || loss : 0.7785 || accuracy : 61.6822
epoch : 20 || loss : 0.6019 || accuracy : 71.0280
epoch : 30 || loss : 0.5252 || accuracy : 75.3894
epoch : 40 || loss : 0.4840 || accuracy : 78.8162
epoch : 50 || loss : 0.4576 || accuracy : 80.5296
epoch : 60 || loss : 0.4388 || accuracy : 81.9315
epoch : 70 || loss : 0.4244 || accuracy : 82.7103
epoch : 80 || loss : 0.4131 || accuracy : 83.1776
epoch : 90 || loss : 0.4039 || accuracy : 83.6449
epoch : 100 || loss : 0.3964 || accuracy : 83.8006
epoch : 110 || loss : 0.3900 || accuracy : 83.8006
epoch : 120 || loss : 0.3846 || accuracy : 83.8006
epoch : 130 || loss : 0.3800 || accuracy : 83.8006
epoch : 140 || loss : 0.3760 || accuracy : 83.8006
epoch : 150 || loss : 0.3725 || accuracy : 83.4891
epoch : 160 || loss : 0.3694 || accuracy : 83.4891
epoch : 170 || loss : 0.3666 || accuracy : 83.4891
epoch : 180 || loss : 0.3642 || accuracy : 83.6449
epoch : 190 || loss : 0.3620 || accuracy :

(0.33549952507019043, 85.202492211838)

After training, we obtained a loss of 0.3332 and an accuracy of 84.89%.

In [16]:
model.evaluate(x_test_tensor,y_test_tesnor)

(0.32219672203063965, 89.4927536231884)

During evaluation, the model achieved a loss of 0.3224 and an accuracy of 88.4%.

# Classification report 

In [17]:
y_predicted = model.predict(x_test_tensor)
y_predicted = torch.round(y_predicted)

In [18]:
print(classification_report(y_test_tesnor.detach().numpy(),y_predicted.detach().numpy()))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88       123
           1       0.89      0.92      0.91       153

    accuracy                           0.89       276
   macro avg       0.90      0.89      0.89       276
weighted avg       0.90      0.89      0.89       276



# ---------------------------------------------------------------------------

## Buliding a neural network with hidden layer (using 3 perceptron)

In [19]:
class Simple_NN:
    def __init__(self,lrt=1e-1):
        self.W1 = None 
        self.W2 = None
        self.B1 = None
        self.B2 = None
        self.lr = lrt
        '''W1 and W2 is the weights of hidden layer.
           B1 and B2 is the Biases of the Hidden layer.
           lr is the learning rate.
        '''
    
    def forward_pass(self,X):
        z1 = torch.matmul(X,self.W1) + self.B1.T
        A1 = torch.relu(z1)

        z2 = torch.matmul(A1,self.W2) + self.B2
        A2 = torch.sigmoid(z2)

        return A2
    

    def log_loss(self, y_true, y_predicted):
        epsilon = 1e-10
    
        # Clamp values to avoid log(0)
        y_predicted = torch.clamp(y_predicted, epsilon, 1 - epsilon)
    
        loss = -torch.mean(y_true * torch.log(y_predicted) + (1 - y_true) * torch.log(1 - y_predicted))
    
        return loss
    



    def fit(self, X, Y, epochs):
        prev_loss = float('inf')
        accuracy = 0

        self.W1 = torch.rand((X.shape[1], 2), dtype = torch.float32, requires_grad = True)
        self.W2 = torch.rand(2, dtype=torch.float32, requires_grad = True)

        self.B1 = torch.zeros((2,1), dtype = torch.float32, requires_grad = True)
        self.B2 = torch.zeros(1, dtype = torch.float32, requires_grad = True)

        for i in range(epochs):

            y_pred = self.forward_pass(X)

            curr_loss = self.log_loss(Y, y_pred)

            y_pred = torch.round(y_pred)
            accuracy = ((y_pred == Y).sum().item() * 100) / len(Y)

            curr_loss.backward()
            
            with torch.no_grad():
                self.W1 -= self.lr * self.W1.grad
                self.W2 -= self.lr * self.W2.grad

                self.B1 -= self.lr * self.B1.grad
                self.B2 -= self.lr * self.B2.grad
            
            self.W1.grad.zero_()
            self.W2.grad.zero_()
            self.B1.grad.zero_()
            self.B2.grad.zero_()

            if (prev_loss - curr_loss) < 1e-5:
                break

            prev_loss = curr_loss

            if i % 10 == 0:
                print(f"epoch : {i} || loss : {curr_loss:.4f} || accuracy : {accuracy:.4f}")

        return accuracy , prev_loss.item()
    

    def predict(self,X):
        return self.forward_pass(X)
    
    def evaluate(self, X, Y):
        y_pred = self.forward_pass(X)
        loss = self.log_loss(Y, y_pred)

        y_pred = torch.round(y_pred)
        accuracy = ((Y == y_pred).sum().item() * 100) / len(Y)

        return accuracy , loss.item()





### Creating the object of the model.

In [20]:
nn = Simple_NN()

In [21]:
nn.fit(x_train_tensor , y_train_tensor,100)

epoch : 0 || loss : 1.0262 || accuracy : 54.0498
epoch : 10 || loss : 0.6578 || accuracy : 58.7227
epoch : 20 || loss : 0.6336 || accuracy : 64.0187
epoch : 30 || loss : 0.6000 || accuracy : 69.1589
epoch : 40 || loss : 0.5597 || accuracy : 73.2087
epoch : 50 || loss : 0.5195 || accuracy : 75.0779
epoch : 60 || loss : 0.4851 || accuracy : 77.4143
epoch : 70 || loss : 0.4581 || accuracy : 78.9720
epoch : 80 || loss : 0.4368 || accuracy : 79.4393
epoch : 90 || loss : 0.4200 || accuracy : 80.3738


(81.15264797507788, 0.40810349583625793)

### After 100 Epochs of Training my model achived :
- Accuracy = 78.81%
- Loss = 0.50


In [22]:
nn.evaluate(x_test_tensor,y_test_tesnor)

(85.14492753623189, 0.37557464838027954)

### During evaluation, the model achieved :
- Accuracy = 83.69%
- Loss = 0.44

# Classification report.

In [23]:
y_pred = nn.predict(x_test_tensor)
y_pred = torch.round(y_pred)
print(classification_report(y_test_tesnor.detach().numpy(),y_pred.detach().numpy()))

              precision    recall  f1-score   support

           0       0.82      0.86      0.84       123
           1       0.88      0.84      0.86       153

    accuracy                           0.85       276
   macro avg       0.85      0.85      0.85       276
weighted avg       0.85      0.85      0.85       276

